# Inversión Cinemática: Terremoto de Copiapó (06-06-2025)

Este notebook realiza la inversión utilizando los parámetros optimizados de axitra y el pre-procesador de datos con control de tiempo y padding.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import os
from obspy import UTCDateTime

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'kdellipspy').exists():
            return p
    raise FileNotFoundError('No root found.')

root = find_project_root(Path(os.getcwd()))
sys.path.append(str(root))
import kdellipspy as kde
print(f"Project root: {root}")

In [ ]:
input_ctl = root / 'inversions' / 'calama2020' / 'input.ctl'
cfg = kde.ConfigParser(filepath=str(input_ctl))
print(f"Evento: {cfg.source_position.event_name} | Estaciones originales: {len(cfg.stations.stations)}")
print("\n=== Configuración Completa ===")
print(f"Nombre del evento: {cfg.source_position.event_name}")
print(f"Tiempo de origen: {cfg.source_position.origin_time}")
print(f"Profundidad: {cfg.source_position.depth}")
print(f"Latitud: {cfg.source_position.latitude}")
print(f"Longitud: {cfg.source_position.longitude}")
print(f"\nDatos observados:")
print(f"  - Número de puntos (npts): {cfg.observed_data.npts}")
print(f"  - Delta (dt): {cfg.observed_data.delta}")
print(f"  - Duración: {cfg.observed_data.npts * cfg.observed_data.delta} s")
print(f"\nEstaciones ({len(cfg.stations.stations)}):")
for s in cfg.stations.stations:
    print(f"  - {s.name}: Lat {s.latitude}, Lon {s.longitude}, Alt {s.height}")

# 2. Visualizacion de parametros de inversion y de fuente


In [ ]:
def mostrar_objeto(obj, nivel=0, max_nivel=2):
    pad = "  " * nivel

    if isinstance(obj, (str, int, float, bool, type(None), np.number)):
        print(f"{pad}{obj!r}")
        return

    if isinstance(obj, (list, tuple)):
        for i, item in enumerate(obj):
            print(f"{pad}[{i}]")
            mostrar_objeto(item, nivel + 1, max_nivel)
        return

    if hasattr(obj, "__dict__") and max_nivel >= 0:
        for k, v in vars(obj).items():
            if k.startswith("_"):
                continue
            if hasattr(v, "__dict__"):
                print(f"{pad}{k}:")
                mostrar_objeto(v, nivel + 1, max_nivel - 1)
            elif isinstance(v, (list, tuple)):
                print(f"{pad}{k}:")
                mostrar_objeto(v, nivel + 1, max_nivel - 1)
            else:
                print(f"{pad}{k}: {v!r}")
    else:
        print(f"{pad}{repr(obj)}")

mostrar_objeto(cfg)

## 4. Inversión Neighbourhood Algorithm (NA)

In [ ]:
output_dir = root / 'inversions' / 'calama2020' / 'DATA'
observed_waveforms, time_array = kde.load_and_filter_observed_data(
    config=cfg,
    data_dir=str(output_dir),
    prefer_raw=False,
)

azi_times_array = kde.build_azi_times_array(config=cfg)

inversion = kde.NAInversionModel(
    config=cfg,
    observed_waveforms=observed_waveforms,
    time_array=time_array,
    azi_times_array=azi_times_array,
    axitra_aw=0.5,
    axitra_ikmax=100000
)

result = inversion.run_na_search()

PLOT RESULTS

In [ ]:
graphics = kde.core.GraphicsSuite(base_dir='./Figures', show=True)
graphics.plot_na_results(result)